<div align="center">

# Hafta 2: Makine öğrenmesi temelleri

**Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı**<br>
Türk Kardiyoloji Derneği · Prof. Dr. Utku Köse

[Hafta sayfası](README.md) · [Ders sayfası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html) · [Etkileşimli lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/lab.html) · [PDF ders notu](Hafta02_Ders_Notlari.pdf) · [Colab'da aç](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-02/NB02_makine_ogrenmesi_temelleri.ipynb)

</div>

**Bu defter iki kısımdan oluşur.** Birincisinde kodu bir yapay zekâ (YZ) asistanı yazar; asistana ne yapacağını yazılı bir talimat, yani bir prompt (istem) anlatır. İkincisinde kodu Python ile kendiniz yazarsınız. İki kısım da aynı verilerle aynı sonuçlara varır; ders sayfasındaki her uygulama durağında bunlardan birini seçin.

| | Kimler için? | Ne yapılır? |
|---|---|---|
| **[Kısım 1: Prompt yolu](#part-1)** | Kod yazmayan katılımcılar | Kodu, hazır promptlarla bir YZ asistanı yazar; siz çalıştırır ve sonucu denetlersiniz. |
| **[Kısım 2: Python yolu](#part-2)** | Python öğrenmek isteyen katılımcılar | Kodu alıştırmalarla adım adım kendiniz yazar ve çalıştırırsınız. |

**Nerede çalışır?** Aynı defter, hiçbir kurulum gerektirmeyen Google Colab'da da kendi bilgisayarınızdaki Jupyter'de de (Anaconda; ders klasöründeki SETUP.tr.md dosyasına bakın) çalışır. Hangisini tercih ederseniz onu kullanın. Colab'da önce *Dosya → Drive'a kopya kaydet* (*File → Save a copy in Drive*) komutuyla kendi kopyanızı kaydedin; aksi hâlde sekme kapanınca yaptığınız değişiklikler kaybolur. Hangi kısmı seçerseniz seçin önce 0. bölümü çalıştırın. Bu bölüm ders dosyalarını ve paketleri hazırlar.

**Veri ve mahremiyet.** Buradaki bütün veri setleri ya sentetiktir ya da açık bir araştırma veri tabanından gelmektedir. Gerçek hasta verisi yalnızca kendi bilgisayarınızda, kurumunuzun kurallarına uygun olarak ve etik kurul onayıyla işlenir; bulut tabanlı bir deftere ya da bir YZ sohbetine asla konmaz.

## 0. Kurulum

İlk hücre ders klasörünü bulur. Defter tek başına açıldıysa, örneğin Google Colab'da, `vclinic` ders paketini ve veri dosyalarını GitHub'dan indirir. İkinci hücre paketleri yükler, mesajların dilini seçer ve haftanın isteğe bağlı paketlerini denetler.

In [ ]:
#@title Ders dosyaları: Bu hücreyi çalıştırın (açmanıza gerek yok)
import pathlib
import sys
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/"
DATA_FILES = [
    "data/uci-heart-disease/processed.cleveland.data",
]


def find_course_root():
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "vclinic" / "__init__.py").exists():
            return folder
    return None


ROOT = find_course_root()
if ROOT is None or ROOT.name == "ai-cardiology-course":
    # İndirilmiş bir kopyada, örneğin Colab'da, paketin dosyaları yenilenir.
    ROOT = pathlib.Path.cwd() / "ai-cardiology-course" if ROOT is None else ROOT
    manifest = urllib.request.urlopen(REPO_RAW + "vclinic/MANIFEST.txt").read().decode("utf-8")
    package = ["vclinic/" + line.strip() for line in manifest.splitlines() if line.strip()]
    for rel in package + DATA_FILES:
        target = ROOT / rel
        if rel in package or not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(REPO_RAW + rel, target)
    print("Ders dosyaları şu klasöre indirildi:", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# Kurulum: paketler, dil ve çizim biçimi
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (adjusted_rand_score, average_precision_score,
                             confusion_matrix, precision_recall_curve,
                             roc_auc_score, roc_curve, silhouette_score)
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from vclinic.datasets import load_cleveland
import vclinic
from vclinic import teach

if tuple(int(x) for x in vclinic.__version__.split(".")) < (1, 1, 0):
    raise ImportError("vclinic ders paketi bu defterden daha eskidir. Ders klasörünü güncelleyin (git pull komutuyla ya da ZIP dosyasını yeniden indirerek) ve çekirdeği yeniden başlatın.")
teach.set_language("tr")
plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans"})
RANDOM_STATE = 2026

print("vclinic sürümü", vclinic.__version__)
print("Ders klasörü:", ROOT)
for name in ["ipywidgets", "lifelines"]:
    try:
        importlib.import_module(name)
        print(f"{name}: kurulu")
    except ImportError:
        print(f"{name}: kurulu değil (isteğe bağlı; defter yerleşik bir yedek yol kullanır)")

In [ ]:
#@title Alıştırmaların beklenen cevapları (bu hücreyi çalıştırın; açmayı alıştırmaları denedikten sonraya bırakın)
teach.register_answers({
    "Alıştırma 2.1": (6, 0),
    "Alıştırma 2.2": (76, 0),
    "Alıştırma 2.3": (0.855, 0.01),
    "Alıştırma A": (0.62, 0.01),
    "Alıştırma B": (0.931, 0.005),
    "Alıştırma C": (0.8, 0.001),
    "Alıştırma D": (0.871, 0.005),
    "Alıştırma E": (0.496, 0.001),
    "Alıştırma F": (0.46, 0.03),
    "Alıştırma G": (0.309, 0.001),
    "Alıştırma H": (0.194, 0.01),
    "Alıştırma I": (0.733, 0.002),
    "Alıştırma J": (0.849, 0.01),
    "Adım 1.1": (0.931, 0.002),
    "Adım 1.2": (0.8, 0.002),
    "Adım 1.3": (0.58, 0.005),
    "Adım 2.1": (0.871, 0.002),
    "Adım 2.2": (0.904, 0.005),
    "Adım 2.3": (0.65, 0.002),
    "Adım 3.1": (0.706, 0.005),
    "Adım 3.2": (0.733, 0.002),
    "Adım 3.3": (0.849, 0.005),
})
print("Beklenen cevaplar hazır.")

<a id="part-1"></a>

# Kısım 1: Prompt yolu

**Kimler için?** Kod yazmayan katılımcılar için. Bu kısımda hiç kod yazmazsınız. Kodu, hazır bir prompttan, yani yazılı bir talimattan yararlanarak bir yapay zekâ (YZ) asistanı yazar; siz kodu çalıştırır ve sonucu denetlersiniz.

**Neye ihtiyacınız var?** Python kodu yazabilen bir YZ asistanına. Google Colab'ın içinde Gemini bulunur; pencerenin üstündeki Gemini düğmesiyle açılır. Başka bir tarayıcı sekmesinde açılan ChatGPT, Claude, Copilot ya da Gemini de aynı işi görür.

**Bir adım nasıl ilerler?**

1. Adımın amacını okuyun.
2. Promptu kopyalayın: Fareyle seçin ve Ctrl+C'ye (Mac'te Cmd+C) basın.
3. Promptu YZ asistanına yapıştırın ve gönderin.
4. Asistanın verdiği kodu kopyalayın, promptun altındaki hücreye `#` ile başlayan satırın yerine yapıştırın ve hücreyi ▶ düğmesiyle ya da Shift+Enter ile çalıştırın.
5. Hemen altındaki kontrol hücresini çalıştırın. Hücre *doğru ✓* ya da *henüz değil ✗* yazar.
6. *Henüz değil* yazarsa aşağıdaki onarım promptunu kodla birlikte asistana gönderin. Sonuç yine yanlışsa *Hazır kod* hücresini çalıştırın. Bu hücre her zaman doğru sonucu verir; böylece bir sonraki adıma geçebilirsiniz.

**Her durağın başında** o durağın hazırlık hücresini bir kez çalıştırın. Hücre durağın verilerini yükler; böylece bir durağa başka bir gün de başlanabilir.

**Onarım promptu** (çalışan ama kontrolden geçmeyen kod için):

> Aşağıdaki kod şu görev için yazıldı: [promptun Görev satırını yapıştırın]. Kod çalışıyor, ancak dersin kontrolü "henüz değil" yazıyor. Hatayı bul, iki cümleyle açıkla ve düzeltilmiş kodu ver. Sonucu tutan değişkenin adını koru.  
>
> [kodu buraya yapıştırın]  

**Hata mesajı için prompt** (hata vererek duran kod için):

> Aşağıdaki kod bir Jupyter defterinde çalıştırıldığında altta verilen hata mesajını veriyor. Hatanın nedenini sade bir dille açıkla ve düzeltilmiş kodu ver.  
>
> [kodu buraya yapıştırın]  
>
> [hata mesajını buraya yapıştırın]  

Haftanın bütün promptları [prompt kütüphanesinde](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-02/PROMPTLAR.md) de yer alır; GitHub her promptun yanında bir kopyalama düğmesi gösterir.

**Hiç değişmeyen kural.** Promptlarda yalnızca dersin sentetik ya da açık verileri bulunur. Bir YZ asistanına gerçek hasta verisi asla yapıştırılmaz.

## Durak 1: İlk risk modeli

*Ders: [1. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html#stop-1)* · *Python yolunda aynı sonuç: Kısım 2, 1-2. bölümler*

UCI Heart Disease veri setindeki Cleveland hastalarına efor testi ve koroner anjiyografi yapılmıştır. Bu durakta öykü ve efor testinden alınan sekiz değişkenle bir lojistik regresyon kurulur; modelin daha önce görmediği hastalarda koroner arter hastalığını ne kadar iyi tanıdığı denetlenir.

**Durağın sonunda:** Test AUC değeri 0,931 olan bir koroner arter hastalığı modeli kuracak, modelin 0,5 eşiğindeki duyarlılığını (0,8) ve efor testinde ulaşılan en yüksek kalp hızının odds oranını (0,58) bulacaksınız.

In [ ]:
#@title 1. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
heart = load_cleveland()
FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
print("1. durak hazır.")

### Adım 1.1: Model ve test AUC değeri

**Amaç.** Bu adımda modelin tablosu kurulur, hastalar eğitim ve test kümelerine ayrılır, bir lojistik regresyon eğitilir ve model test kümesinde ROC eğrisinin altındaki alanla (AUC) ölçülür.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazdığın kod, bir Jupyter defterindeki hücreye yapıştırılıp orada çalıştırılacak.  
> Bağlam: Kod, aşağıdaki verilerin önceden yüklendiği 2. hafta Jupyter defterinde çalışacak. Tablolar, hesaplar ve grafikler için kullanılan paketler (pandas, NumPy ve Matplotlib) ile makine öğrenmesi paketi scikit-learn hazırdır. RANDOM_STATE, defterin rastlantının rol oynadığı her yerde kullandığı sabit değeri tutar.  
> Veri: `heart`, UCI Heart Disease veri setinde Cleveland Clinic'ten gelen 303 hastayı tutan bir tablodur ve her hasta bir satırda yer alır. Burada kullanılan sütunlar şunlardır: age (yaş, yıl); sex (erkekler için 1, kadınlar için 0); göğüs ağrısının tipini veren cp (1 tipik anjina, 2 atipik anjina, 3 anjina dışı ağrı, 4 asemptomatik); trestbps (istirahat kan basıncı, mmHg); chol (serum kolesterolü, mg/dL); thalach (efor testinde ulaşılan en yüksek kalp hızı, atım/dk); exang (eforla ortaya çıkan anjina; varsa 1); oldpeak (eforda istirahate göre ST çökmesi, mm) ve disease. disease, anjiyografide en az bir büyük koroner arterin çapı yüzde 50'den fazla daralmışsa 1, değilse 0 değerini alır.  
> Görev: Modelin X tablosunu age, sex, trestbps, chol, thalach, exang, oldpeak ve cp sütunlarından kur. Göğüs ağrısı tipi (cp) bir kategoridir: Onu her tip için bir evet/hayır sütununa çevir ve ilk tipi, yani tipik anjinayı, referans olarak dışarıda bırak. Sonuç değişkeni y, disease sütunudur. Hastaları dörtte üçü eğitim, dörtte biri test kümesi olacak biçimde rastgele ayır; hastalığı olanların oranı iki kümede de aynı kalsın ve rastgele ayırma için RANDOM_STATE değerini kullan. Değişkenleri yalnızca eğitim hastalarının ortalama ve standart sapmalarıyla standartlaştır, eğitim kümesinde bir lojistik regresyon eğit ve modelin test kümesindeki olasılıklarından ROC eğrisinin altındaki alanı (AUC) hesapla.  
> Kısıtlar: Yalnızca burada adı geçen verileri ve araçları kullan. Kod dosya okumamalı, bir şey indirmemeli ve paket kurmamalıdır.  
> Çıktı: Test kümesinin AUC değerini üç ondalık basamağa yuvarlayarak ver. X, y, X_train, X_test, y_train ve y_test tablolarını ve eğitilmiş modeli model adıyla sakla; sonraki adımlar bunları kullanmaktadır. Sonucu `auc_test` adıyla sakla ve kısa bir açıklamayla birlikte yazdır. Yalnızca kodu tek bir blok hâlinde ver ve her adımın üstüne kısa bir açıklama satırı ekle.  
> Test: Test kümesinde 76 hasta bulunmalı ve AUC 0,5 ile 1 arasında olmalıdır.  

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.1 kontrolü
teach.check_step("Adım 1.1", "auc_test", globals())

In [ ]:
#@title Adım 1.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
heart = load_cleveland()
FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
data = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"], drop_first=True, dtype=int)
X = data.drop(columns="disease")
y = data["disease"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)
auc_test = round(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]), 3)
print("Test AUC:", auc_test)
teach.check_step("Adım 1.1", "auc_test", globals())

**Sonuç ne gösteriyor?** Model, rastgele seçilen bir koroner arter hastasını rastgele seçilen ve hastalığı olmayan bir kişinin önüne her 100 çiftin yaklaşık 93 tanesinde yerleştirir. 76 hastalık bir test kümesi bu kestirimi yine de belirsiz bırakır; 2. durak testi başka bir yolla tekrarlamaktadır.

### Adım 1.2: 0,5 eşiğinde duyarlılık

**Amaç.** Bu adımda olasılıklar 0,5 eşiğinde karara dönüştürülür ve modelin hastalığı olanların ne kadarını tanıdığı bulunur.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazdığın kod, bir Jupyter defterindeki hücreye yapıştırılıp orada çalıştırılacak.  
> Bağlam: Kod, aşağıdaki verilerin önceden yüklendiği 2. hafta Jupyter defterinde çalışacak. Tablolar, hesaplar ve grafikler için kullanılan paketler (pandas, NumPy ve Matplotlib) ile makine öğrenmesi paketi scikit-learn hazırdır. RANDOM_STATE, defterin rastlantının rol oynadığı her yerde kullandığı sabit değeri tutar.  
> Veri: 1.1. adımda eğitilmiş model (model) ile X_test ve y_test test kümesi (y_test: hastalık varsa 1, yoksa 0).  
> Görev: Modelin her test hastasına verdiği hastalık olasılığını hesapla ve olasılığı 0,5 ya da üzerinde olan hastayı pozitif say. Doğru pozitifleri (TP; hastalığı olup pozitif sayılanlar) ve yanlış negatifleri (FN; hastalığı olup negatif sayılanlar) say. Duyarlılık, TP / (TP + FN) oranıdır.  
> Kısıtlar: Yalnızca burada adı geçen verileri ve araçları kullan. Kod dosya okumamalı, bir şey indirmemeli ve paket kurmamalıdır.  
> Çıktı: Duyarlılığı üç ondalık basamağa yuvarlayarak ver. Sonucu `sensitivity` adıyla sakla ve kısa bir açıklamayla birlikte yazdır. Yalnızca kodu tek bir blok hâlinde ver ve her adımın üstüne kısa bir açıklama satırı ekle.  
> Test: TP + FN, test kümesindeki hastalığı olanların sayısına, yani 35'e eşit olmalıdır.  

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.2 kontrolü
teach.check_step("Adım 1.2", "sensitivity", globals())

In [ ]:
#@title Adım 1.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
p_test = model.predict_proba(X_test)[:, 1]
pred = (p_test >= 0.5).astype(int)
tp = int(((pred == 1) & (y_test == 1)).sum())
fn = int(((pred == 0) & (y_test == 1)).sum())
sensitivity = round(tp / (tp + fn), 3)
print("TP =", tp, " FN =", fn, " duyarlılık =", sensitivity)
teach.check_step("Adım 1.2", "sensitivity", globals())

**Sonuç ne gösteriyor?** Bu eşikte model, hastalığı olan yaklaşık her beş hastadan birini kaçırır. Daha düşük bir eşik bu hastaların daha fazlasını bulur, ancak daha çok yanlış alarm verir; etkileşimli labın B bölümü bu dengeyi göstermektedir.

### Adım 1.3: En yüksek kalp hızının odds oranı

**Amaç.** Bu adımda efor testinde ulaşılan en yüksek kalp hızı bir standart sapma arttığında koroner arter hastalığının odds değerinin nasıl değiştiği modelden okunur.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazdığın kod, bir Jupyter defterindeki hücreye yapıştırılıp orada çalıştırılacak.  
> Bağlam: Kod, aşağıdaki verilerin önceden yüklendiği 2. hafta Jupyter defterinde çalışacak. Tablolar, hesaplar ve grafikler için kullanılan paketler (pandas, NumPy ve Matplotlib) ile makine öğrenmesi paketi scikit-learn hazırdır. RANDOM_STATE, defterin rastlantının rol oynadığı her yerde kullandığı sabit değeri tutar.  
> Veri: 1.1. adımda eğitim kümesinde eğitilmiş model (model): standartlaştırılmış değişkenler üzerinde bir lojistik regresyon. Değişkenlerin adları X tablosunun sütun adlarıdır.  
> Görev: Lojistik regresyonun katsayılarını al, her birini ait olduğu değişkenin adıyla eşleştir ve her katsayıyı standart sapma başına bir odds oranına, yani e'nin katsayı kadar kuvvetine çevir. Bütün odds oranlarını yazdır.  
> Kısıtlar: Yalnızca burada adı geçen verileri ve araçları kullan. Kod dosya okumamalı, bir şey indirmemeli ve paket kurmamalıdır.  
> Çıktı: thalach değişkeninin odds oranını iki ondalık basamağa yuvarlayarak ver. Sonucu `or_thalach` adıyla sakla ve kısa bir açıklamayla birlikte yazdır. Yalnızca kodu tek bir blok hâlinde ver ve her adımın üstüne kısa bir açıklama satırı ekle.  
> Test: oldpeak değişkeninin odds oranı 1'in üzerinde olmalıdır.  

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.3 kontrolü
teach.check_step("Adım 1.3", "or_thalach", globals())

In [ ]:
#@title Adım 1.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
lr = model.named_steps["logisticregression"]
odds = pd.Series(np.exp(lr.coef_[0]), index=X.columns)
print(odds.round(2))
or_thalach = round(float(odds["thalach"]), 2)
print("thalach için SS başına odds oranı:", or_thalach)
teach.check_step("Adım 1.3", "or_thalach", globals())

**Sonuç ne gösteriyor?** Odds oranı 1'in altındadır: Efor testinde daha yüksek bir kalp hızı, koroner arter hastalığının daha düşük odds değeriyle birlikte görülür; klinikte iyi bir kronotropik yanıttan beklenen de budur. Oran, dakikada bir atıma değil, eğitim hastalarında thalach değişkeninin bir standart sapmasına karşılık gelmektedir.

**1. durak tamamlandı.** [Ders sayfasına](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html#stop-1) dönün ve duraktan sonrasını okumaya devam edin.

## Durak 2: Bir modeli dürüstçe sınamak

*Ders: [2. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html#stop-2)* · *Python yolunda aynı sonuç: Kısım 2, 3-6. bölümler*

76 hastalık tek bir test kümesi belirsiz bir tahmin verir ve bazı modeller olduklarından daha iyi görünür. Bu durakta test çapraz doğrulamayla tekrarlanır ve bir rastgele orman denenir. Ardından sanal kliniğin nadir bir sonucuna geçilir: 12 ay içinde ölüm ya da kalp yetersizliği nedeniyle hastaneye yatış.

**Durağın sonunda:** Lojistik regresyonun çapraz doğrulanmış AUC değerini (0,871), bir rastgele ormanın test AUC değerini (0,904) ve sınıf ağırlıklarıyla eğitilen nadir 12 aylık olay modelinin duyarlılığını (0,65) elde edeceksiniz.

In [ ]:
#@title 2. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
heart = load_cleveland()
FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
data = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"], drop_first=True, dtype=int)
X = data.drop(columns="disease")
y = data["disease"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
registry = vclinic.add_followup(vclinic.make_cohort(2000, seed=RANDOM_STATE))
registry["event_12m"] = ((registry["event"] == 1) & (registry["followup_months"] <= 12)).astype(int)
print("2. durak hazır.")

### Adım 2.1: Beş katlı çapraz doğrulama

**Amaç.** Bu adımda 1. duraktaki lojistik regresyon, her seferinde hastaların başka bir beşte birinde olmak üzere beş kez sınanır ve ortalama AUC alınır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazdığın kod, bir Jupyter defterindeki hücreye yapıştırılıp orada çalıştırılacak.  
> Bağlam: Kod, aşağıdaki verilerin önceden yüklendiği 2. hafta Jupyter defterinde çalışacak. Tablolar, hesaplar ve grafikler için kullanılan paketler (pandas, NumPy ve Matplotlib) ile makine öğrenmesi paketi scikit-learn hazırdır. RANDOM_STATE, defterin rastlantının rol oynadığı her yerde kullandığı sabit değeri tutar.  
> Veri: 1.1. adımın X ve y tabloları: X, 303 Cleveland hastasının tamamı için modelin değişkenlerini; y ise sonucu, yani hastalık varsa 1, yoksa 0 değerini tutar.  
> Görev: 1.1. adımdakiyle aynı türden bir modeli, yani standartlaştırma ve ardından lojistik regresyonu, beş katlı çapraz doğrulamayla değerlendir. Hastaları RANDOM_STATE ile karıştır ve hastalığı olanların oranını koruyan beş kata ayır. Dört katta eğit, beşincide AUC değerini hesapla; her kat bir kez test kümesi olacak biçimde bunu beş kez tekrarla. Beş AUC değerini ve ortalamalarını yazdır.  
> Kısıtlar: Yalnızca burada adı geçen verileri ve araçları kullan. Kod dosya okumamalı, bir şey indirmemeli ve paket kurmamalıdır.  
> Çıktı: Beş katın ortalama AUC değerini üç ondalık basamağa yuvarlayarak ver. Sonucu `auc_cv` adıyla sakla ve kısa bir açıklamayla birlikte yazdır. Yalnızca kodu tek bir blok hâlinde ver ve her adımın üstüne kısa bir açıklama satırı ekle.  
> Test: Her biri 0,5 ile 1 arasında olan beş değer yazdırılmalıdır.  

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.1 kontrolü
teach.check_step("Adım 2.1", "auc_cv", globals())

In [ ]:
#@title Adım 2.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
pipeline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
folds = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(pipeline, X, y, cv=folds, scoring="roc_auc")
print("Beş katın AUC değerleri:", scores.round(3))
auc_cv = round(scores.mean(), 3)
print("Ortalama AUC:", auc_cv)
teach.check_step("Adım 2.1", "auc_cv", globals())

**Sonuç ne gösteriyor?** Ortalama, 1.1. adımdaki tek test kümesinin AUC değerinden düşüktür ve beş değer birbirinden epeyce farklıdır. 303 hastalık bir tabloda sonuç, test kümesine hangi hastaların düştüğüne belirgin biçimde bağlıdır.

### Adım 2.2: Bir rastgele orman

**Amaç.** Bu adımda aynı hastalarda daha esnek bir model eğitilir ve test AUC değeri lojistik regresyonunkiyle karşılaştırılır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazdığın kod, bir Jupyter defterindeki hücreye yapıştırılıp orada çalıştırılacak.  
> Bağlam: Kod, aşağıdaki verilerin önceden yüklendiği 2. hafta Jupyter defterinde çalışacak. Tablolar, hesaplar ve grafikler için kullanılan paketler (pandas, NumPy ve Matplotlib) ile makine öğrenmesi paketi scikit-learn hazırdır. RANDOM_STATE, defterin rastlantının rol oynadığı her yerde kullandığı sabit değeri tutar.  
> Veri: 1.1. adımın tabloları: X, modelin değişkenlerini, yani age, sex, trestbps, chol, thalach, exang, oldpeak sütunlarını ve göğüs ağrısı tiplerinin evet/hayır sütunlarını; y ise disease sütununu tutar. X_train ve y_train eğitim kümesini, X_test ve y_test ise 303 hastanın 76'sından oluşan test kümesini oluşturur.  
> Görev: Eğitim kümesinde 300 ağaçlı bir rastgele orman eğit; ormanın rastgele seçimleri için RANDOM_STATE değerini kullan. Ardından ormanın test kümesindeki olasılıklarından AUC değerini hesapla.  
> Kısıtlar: Yalnızca burada adı geçen verileri ve araçları kullan. Kod dosya okumamalı, bir şey indirmemeli ve paket kurmamalıdır.  
> Çıktı: Ormanın test AUC değerini üç ondalık basamağa yuvarlayarak ver. Sonucu `auc_forest` adıyla sakla ve kısa bir açıklamayla birlikte yazdır. Yalnızca kodu tek bir blok hâlinde ver ve her adımın üstüne kısa bir açıklama satırı ekle.  
> Test: AUC 0,5 ile 1 arasında olmalıdır.  

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.2 kontrolü
teach.check_step("Adım 2.2", "auc_forest", globals())

In [ ]:
#@title Adım 2.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
forest = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)
forest.fit(X_train, y_train)
auc_forest = round(roc_auc_score(y_test, forest.predict_proba(X_test)[:, 1]), 3)
print("Ormanın test AUC değeri:", auc_forest)
teach.check_step("Adım 2.2", "auc_forest", globals())

**Sonuç ne gösteriyor?** Orman bu hastalarda lojistik regresyonu geçemez. Esnek modeller çok sayıda hastaya ihtiyaç duyar; küçük bir tabloda basit ve okunabilir bir model çoğu zaman aynı ölçüde iyidir.

### Adım 2.3: Nadir bir sonuç

**Amaç.** Bu adımda hastaların yalnızca 100'de 3 kadarında görülen 12 aylık olay için bir model eğitilir ve sınıf ağırlıklarının modelin bulduklarını nasıl değiştirdiği görülür.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazdığın kod, bir Jupyter defterindeki hücreye yapıştırılıp orada çalıştırılacak.  
> Bağlam: Kod, aşağıdaki verilerin önceden yüklendiği 2. hafta Jupyter defterinde çalışacak. Tablolar, hesaplar ve grafikler için kullanılan paketler (pandas, NumPy ve Matplotlib) ile makine öğrenmesi paketi scikit-learn hazırdır. RANDOM_STATE, defterin rastlantının rol oynadığı her yerde kullandığı sabit değeri tutar.  
> Veri: `registry`, sanal kliniğin 2.000 sentetik hastasını izlem bilgileriyle birlikte tutan bir tablodur ve her hasta bir satırda yer alır. Burada kullanılan sütunlar şunlardır: age (yaş, yıl), egfr (tahmini glomerüler filtrasyon hızı, mL/dk/1,73 m²), lvef (sol ventrikül ejeksiyon fraksiyonu, %), ntprobnp (N-terminal pro-B tipi natriüretik peptit, pg/mL), af (atriyal fibrilasyon; varsa 1), diabetes (varsa 1), bmi (beden kitle indeksi, kg/m²), sbp (sistolik kan basıncı, mmHg) ve event_12m. event_12m, hasta izlemin ilk 12 ayında ölmüşse ya da kalp yetersizliği nedeniyle hastaneye yatmışsa 1, değilse 0'dır.  
> Görev: age, egfr, lvef, ntprobnp, af ve diabetes sütunlarından Xe tablosunu kur; ntprobnp yerine onun doğal logaritmasını kullan. Sonuç değişkeni ye, event_12m sütunudur. Hastaları RANDOM_STATE ile ve olay oranını koruyarak yüzde 70'i eğitim, yüzde 30'u test kümesi olacak biçimde rastgele ayır. Değişkenleri eğitim hastalarıyla standartlaştır ve dengeli sınıf ağırlıklarıyla bir lojistik regresyon eğit; bu ağırlıklar, olay yaşayan az sayıdaki hastaya toplamda diğer bütün hastalar kadar ağırlık verir. Olasılığı 0,5 ya da üzerinde olan test hastasını pozitif say ve test kümesinde duyarlılığı, yani TP / (TP + FN) oranını hesapla.  
> Kısıtlar: Yalnızca burada adı geçen verileri ve araçları kullan. Kod dosya okumamalı, bir şey indirmemeli ve paket kurmamalıdır.  
> Çıktı: Duyarlılığı üç ondalık basamağa yuvarlayarak ver. Sonucu `se_balanced` adıyla sakla ve kısa bir açıklamayla birlikte yazdır. Yalnızca kodu tek bir blok hâlinde ver ve her adımın üstüne kısa bir açıklama satırı ekle.  
> Test: Test kümesinde 600 hasta bulunmalı ve bunların 20'sinde olay olmalıdır. Sınıf ağırlıkları olmayan aynı model, 0,5 eşiğinde bu olayların hiçbirini bulamaz.  

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.3 kontrolü
teach.check_step("Adım 2.3", "se_balanced", globals())

In [ ]:
#@title Adım 2.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
registry = vclinic.add_followup(vclinic.make_cohort(2000, seed=RANDOM_STATE))
registry["event_12m"] = ((registry["event"] == 1) & (registry["followup_months"] <= 12)).astype(int)
Xe = registry[["age", "egfr", "lvef", "ntprobnp", "af", "diabetes"]].assign(ntprobnp=np.log(registry["ntprobnp"]))
ye = registry["event_12m"]
Xe_train, Xe_test, ye_train, ye_test = train_test_split(Xe, ye, test_size=0.3, stratify=ye, random_state=RANDOM_STATE)
weighted = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000))
weighted.fit(Xe_train, ye_train)
pred = (weighted.predict_proba(Xe_test)[:, 1] >= 0.5).astype(int)
tp = int(((pred == 1) & (ye_test == 1)).sum())
fn = int(((pred == 0) & (ye_test == 1)).sum())
se_balanced = round(tp / (tp + fn), 3)
print("Test kümesindeki olaylar:", int(ye_test.sum()), " duyarlılık =", se_balanced)
teach.check_step("Adım 2.3", "se_balanced", globals())

**Sonuç ne gösteriyor?** Sınıf ağırlıkları olmadan herkese olay yok diyen bir model zaten yüzde 97 doğru sonuç verir ve olayların hiçbirini bulmaz. Ağırlıklar, nadir olayların eğitimde daha çok sayılmasını sağlar; bunun bedeli düşük bir pozitif öngörü değeridir, yani az sayıda olay için çok sayıda alarm.

**2. durak tamamlandı.** [Ders sayfasına](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html#stop-2) dönün ve duraktan sonrasını okumaya devam edin.

## Durak 3: Alt gruplar, sağkalım ve denetlenecek bir model

*Ders: [3. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html#stop-3)* · *Python yolunda aynı sonuç: Kısım 2, 7-9. bölümler*

Kalp yetersizliği olan hastalar yalnızca ejeksiyon fraksiyonuyla değil, başka özellikleriyle de birbirinden ayrılır. Bu durakta önce, yönteme sonuç bilgisi verilmeden alt gruplar aranır; ardından izlem boyunca sağkalım kestirilir. Son olarak bir YZ asistanının kusursuz bir sonuç bildirdiği bir model denetlenir.

**Durağın sonunda:** Üç kümenin kliniğin gizli fenogruplarıyla uyumunu (düzeltilmiş Rand indeksi 0,706), HFrEF hastalarının 24. aydaki olaysız sağkalımını (0,733) ve denetlenen modelin dürüst AUC değerini (0,849) elde edeceksiniz.

In [ ]:
#@title 3. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
registry = vclinic.add_followup(vclinic.make_cohort(2000, seed=RANDOM_STATE))
registry["event_12m"] = ((registry["event"] == 1) & (registry["followup_months"] <= 12)).astype(int)
hf = registry[registry["main_diagnosis"].isin(["HFrEF", "HFpEF"])].copy()
HF_FEATURES = ["age", "bmi", "egfr", "lvef", "ntprobnp", "lavi", "e_e_prime", "lv_mass_index", "tapse_mm"]
print("3. durak hazır.")

### Adım 3.1: Üç küme

**Amaç.** Bu adımda k-ortalamalar yöntemi kalp yetersizliği olan hastalar arasında üç grup arar; ardından bu gruplar gizli fenogruplarla karşılaştırılır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazdığın kod, bir Jupyter defterindeki hücreye yapıştırılıp orada çalıştırılacak.  
> Bağlam: Kod, aşağıdaki verilerin önceden yüklendiği 2. hafta Jupyter defterinde çalışacak. Tablolar, hesaplar ve grafikler için kullanılan paketler (pandas, NumPy ve Matplotlib) ile makine öğrenmesi paketi scikit-learn hazırdır. RANDOM_STATE, defterin rastlantının rol oynadığı her yerde kullandığı sabit değeri tutar.  
> Veri: `hf`, sanal klinikte kalp yetersizliği olan 407 hastayı, yani ana tanısı HFrEF ya da HFpEF olanları tutan bir tablodur. `HF_FEATURES` bu tablonun dokuz sütununu sıralar: age, bmi, egfr, lvef, ntprobnp, lavi (sol atriyum hacim indeksi, mL/m²), e_e_prime (E/e′ oranı), lv_mass_index (sol ventrikül kütle indeksi, g/m²) ve tapse_mm (triküspit anüler düzlem sistolik hareketi, mm). hf_phenogroup sütunu, kliniğin verileri üretirken kullandığı gizli fenogrubu tutar.  
> Görev: HF_FEATURES listesindeki dokuz sütunu al, ntprobnp yerine onun doğal logaritmasını kullan ve sütunları standartlaştır. Hastaları k-ortalamalar yöntemiyle üç kümeye ayır; yöntemi RANDOM_STATE ile ve farklı rastgele başlangıç noktalarından on kez başlat. Kümeleri gizli fenogruplarla (hf_phenogroup) düzeltilmiş Rand indeksiyle karşılaştır.  
> Kısıtlar: Yalnızca burada adı geçen verileri ve araçları kullan. Kod dosya okumamalı, bir şey indirmemeli ve paket kurmamalıdır.  
> Çıktı: Düzeltilmiş Rand indeksini üç ondalık basamağa yuvarlayarak ver. Sonucu `ari` adıyla sakla ve kısa bir açıklamayla birlikte yazdır. Yalnızca kodu tek bir blok hâlinde ver ve her adımın üstüne kısa bir açıklama satırı ekle.  
> Test: Üç kümedeki hastaların toplamı 407 olmalıdır. İndeksin 1 olması tam uyumu, 0 olması ise rastlantısal uyumu gösterir.  

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.1 kontrolü
teach.check_step("Adım 3.1", "ari", globals())

In [ ]:
#@title Adım 3.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
X_hf = StandardScaler().fit_transform(hf[HF_FEATURES].assign(ntprobnp=np.log(hf["ntprobnp"])))
km3 = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE).fit(X_hf)
print("Kümelerdeki hasta sayıları:", np.bincount(km3.labels_))
ari = round(adjusted_rand_score(hf["hf_phenogroup"], km3.labels_), 3)
print("Düzeltilmiş Rand indeksi:", ari)
teach.check_step("Adım 3.1", "ari", globals())

**Sonuç ne gösteriyor?** Kümeler gizli yapının büyük bölümünü yakalar, ama tamamını değil. Gerçek verilerde doğru grupları kimse bilmez: Kümeler, bir anlam kazanmadan önce klinik muhakemeyle ve sonuç verileriyle sınanması gereken hipotezlerdir.

### Adım 3.2: 24. ayda Kaplan-Meier

**Amaç.** Bu adımda HFrEF hastalarının izlemin başlamasından iki yıl sonra hangi oranda hayatta olduğu ve kalp yetersizliği nedeniyle hastaneye yatmadığı kestirilir.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazdığın kod, bir Jupyter defterindeki hücreye yapıştırılıp orada çalıştırılacak.  
> Bağlam: Kod, aşağıdaki verilerin önceden yüklendiği 2. hafta Jupyter defterinde çalışacak. Tablolar, hesaplar ve grafikler için kullanılan paketler (pandas, NumPy ve Matplotlib) ile makine öğrenmesi paketi scikit-learn hazırdır. RANDOM_STATE, defterin rastlantının rol oynadığı her yerde kullandığı sabit değeri tutar.  
> Veri: `hf`, sanal klinikte kalp yetersizliği olan 407 hastayı, yani ana tanısı HFrEF ya da HFpEF olanları tutan bir tablodur. `HF_FEATURES` bu tablonun dokuz sütununu sıralar: age, bmi, egfr, lvef, ntprobnp, lavi (sol atriyum hacim indeksi, mL/m²), e_e_prime (E/e′ oranı), lv_mass_index (sol ventrikül kütle indeksi, g/m²) ve tapse_mm (triküspit anüler düzlem sistolik hareketi, mm). hf_phenogroup sütunu, kliniğin verileri üretirken kullandığı gizli fenogrubu tutar. followup_months, izlemin başlangıcından ilk olaya ya da izlemin sonuna kadar geçen süredir (ay). event, ölüm ya da kalp yetersizliği nedeniyle yatış için 1, izlemden olaysız ayrılan (sansürlenen) hasta için 0'dır.  
> Görev: Ana tanısı HFrEF olan hastalar için olaysız sağkalımın Kaplan-Meier kestirimini adım adım hesapla. 1'den başla. En az bir olayın görüldüğü her zamanda kestirimi 1 − d/n ile çarp; burada d o zamandaki olay sayısı, n ise o zamanda hâlâ izlenen, yani izlem süresi en az o kadar olan hasta sayısıdır. 24. aydaki kestirim, 24 ay ya da daha kısa olan son olay zamanından sonraki değerdir.  
> Kısıtlar: Yalnızca burada adı geçen verileri ve araçları kullan. Kod dosya okumamalı, bir şey indirmemeli ve paket kurmamalıdır.  
> Çıktı: 24. aydaki kestirimi üç ondalık basamağa yuvarlayarak ver. Sonucu `s24_hfref` adıyla sakla ve kısa bir açıklamayla birlikte yazdır. Yalnızca kodu tek bir blok hâlinde ver ve her adımın üstüne kısa bir açıklama satırı ekle.  
> Test: Aynı hesap HFpEF hastaları için 24. ayda daha yüksek bir değer vermelidir.  

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.2 kontrolü
teach.check_step("Adım 3.2", "s24_hfref", globals())

In [ ]:
#@title Adım 3.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
def km_at(time, event, t_end):
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    s = 1.0
    for t in np.unique(time[event == 1]):
        if t > t_end:
            break
        s *= 1 - np.sum((time == t) & (event == 1)) / np.sum(time >= t)
    return s


ref = hf[hf["main_diagnosis"] == "HFrEF"]
pef = hf[hf["main_diagnosis"] == "HFpEF"]
s24_hfref = round(km_at(ref["followup_months"], ref["event"], 24), 3)
print("S(24), HFrEF:", s24_hfref, " HFpEF:", round(km_at(pef["followup_months"], pef["event"], 24), 3))
teach.check_step("Adım 3.2", "s24_hfref", globals())

**Sonuç ne gösteriyor?** Bu sentetik kayıtta HFrEF hastalarının yaklaşık dörtte üçü iki yılın sonunda hayattadır ve kalp yetersizliği nedeniyle hastaneye yatmamıştır. Sansürlenen hastalar izlemden ayrılana kadar risk altında sayılır; onları dışarıda bırakmak kestirimi saptırır.

### Adım 3.3: Bir asistanın modelini denetlemek

**Amaç.** Bir YZ asistanı, 12 ay içindeki olay için 1,000'lik bir AUC bildirmiştir. Bu adımda aynı türden bir modelin dürüst bir değerlendirmede ne verdiği bulunur.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazdığın kod, bir Jupyter defterindeki hücreye yapıştırılıp orada çalıştırılacak.  
> Bağlam: Kod, aşağıdaki verilerin önceden yüklendiği 2. hafta Jupyter defterinde çalışacak. Tablolar, hesaplar ve grafikler için kullanılan paketler (pandas, NumPy ve Matplotlib) ile makine öğrenmesi paketi scikit-learn hazırdır. RANDOM_STATE, defterin rastlantının rol oynadığı her yerde kullandığı sabit değeri tutar.  
> Veri: `registry`, sanal kliniğin 2.000 sentetik hastasını izlem bilgileriyle birlikte tutan bir tablodur ve her hasta bir satırda yer alır. Burada kullanılan sütunlar şunlardır: age (yaş, yıl), egfr (tahmini glomerüler filtrasyon hızı, mL/dk/1,73 m²), lvef (sol ventrikül ejeksiyon fraksiyonu, %), ntprobnp (N-terminal pro-B tipi natriüretik peptit, pg/mL), af (atriyal fibrilasyon; varsa 1), diabetes (varsa 1), bmi (beden kitle indeksi, kg/m²), sbp (sistolik kan basıncı, mmHg) ve event_12m. event_12m, hasta izlemin ilk 12 ayında ölmüşse ya da kalp yetersizliği nedeniyle hastaneye yatmışsa 1, değilse 0'dır. Asistanın modeli, ancak izlemden sonra bilinen followup_months değişkenini de kullanmış ve AUC değerini eğitildiği hastaların kendisinde ölçmüştür.  
> Görev: Karar anında bilinmediği için followup_months değişkenini dışarıda bırak. event_12m sonucunu age, egfr, lvef, ntprobnp, af, diabetes, bmi ve sbp sütunlarıyla, yaprakları en az 5 hasta içeren 200 ağaçlı bir rastgele ormanla öngör; ormanda RANDOM_STATE değerini kullan. Modeli, hastaları RANDOM_STATE ile karıştıran ve her katta olay oranını koruyan beş katlı bir çapraz doğrulamayla ölç ve beş katın ortalama AUC değerini al.  
> Kısıtlar: Yalnızca burada adı geçen verileri ve araçları kullan. Kod dosya okumamalı, bir şey indirmemeli ve paket kurmamalıdır.  
> Çıktı: Ortalama AUC değerini üç ondalık basamağa yuvarlayarak ver. Sonucu `auc_review` adıyla sakla ve kısa bir açıklamayla birlikte yazdır. Yalnızca kodu tek bir blok hâlinde ver ve her adımın üstüne kısa bir açıklama satırı ekle.  
> Test: AUC, 1,000'in belirgin biçimde altında olmalıdır.  

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.3 kontrolü
teach.check_step("Adım 3.3", "auc_review", globals())

In [ ]:
#@title Adım 3.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
columns = ["age", "egfr", "lvef", "ntprobnp", "af", "diabetes", "bmi", "sbp"]
forest_cv = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=RANDOM_STATE)
folds = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
auc_review = round(cross_val_score(forest_cv, registry[columns], registry["event_12m"], cv=folds, scoring="roc_auc").mean(), 3)
print("followup_months olmadan çapraz doğrulanmış AUC:", auc_review)
teach.check_step("Adım 3.3", "auc_review", globals())

**Sonuç ne gösteriyor?** Bu, nadir bir sonuç için gerçekçi bir değerdir. Kusursuz bir AUC bir uyarı işaretidir: Ya izlem süresi gibi gelecekten gelen bir bilgi modele girmiştir ya da test, modelin eğitildiği hastalarda yapılmıştır.

**3. durak tamamlandı.** [Ders sayfasına](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html#stop-3) dönün ve duraktan sonrasını okumaya devam edin.

<a id="part-2"></a>

# Kısım 2: Python yolu

**Kimler için?** Python yazmayı öğrenmek isteyen katılımcılar için. Bu kısım için Kısım 1'i çalışmış olmak gerekmez: 0. bölümden sonra doğrudan buradan başlayın.

**Nasıl ilerler?** Hücreleri yukarıdan aşağıya Shift+Enter ile çalıştırın. Baştaki *Python adımı*, haftanın gerektirdiği dil bilgisini öğretir. Numaralı bölümlerdeki alıştırmalar hemen *doğru* ya da *henüz değil* yazar; örnek çözümler katlanmış hücrelerde durur ve ancak denedikten sonra açılmalıdır. *Gözlemle ve düşün* hücreleri açık uçlu sorular sorar; bu soruların çözüm hücresi yoktur. *Python notu* hücreleri, yeni bir yapının ilk kullanıldığı yerde dili açıklar. *İleri düzey (isteğe bağlı)* olarak işaretlenen hücreler, programlama bilenler için ek çalışmalardır. Bu kısma ortadan başlayan katılımcı, o bölümün başındaki *Telafi* hücresini çalıştırır.

## Python adımı 2: Veriyi modele hazırlamak

scikit-learn'deki bir model iki şey ister: Her satırı bir hasta olan ve `X` adı verilen bir sayı tablosu ile her hastanın sonucunu tutan `y`. Bu adım, 1. haftadaki Cleveland verisini böyle bir tabloya dönüştürür. Sütun ve satır seçer, eksik değerleri ele alır, bir kategoriyi sayılara çevirir, hastaları eğitim ve test kümelerine ayırır ve ilk modeli kurar. Defterin sonraki bütün bölümleri bu adımları kullanmaktadır.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
heart = load_cleveland()
print(heart.shape)
heart[["age", "chol", "disease"]].head()

**Python notu.** `heart.shape` satır ve sütun sayısını verir. Çift köşeli parantez (`heart[["age", "chol"]]`) birden fazla sütun seçer ve daha küçük bir DataFrame döndürür. Tek köşeli parantez (`heart["age"]`) ise tek bir sütun seçer ve aynı satır etiketlerini taşıyan tek sütunluk bir *Series* döndürür.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
older = heart[heart["age"] >= 60]
print(len(older), "hasta 60 yaşında ya da daha yaşlı")
heart.loc[heart["sex"] == 0, ["age", "chol", "disease"]].head(3)

**Python notu.** `heart["age"] >= 60` her yaşı 60 ile karşılaştırır ve `True` ile `False` değerlerinden oluşan bir sütun, yani bir *Boolean maske* verir. Maske köşeli parantez içine yazıldığında yalnızca `True` olan satırlar kalır. `.loc[satırlar, sütunlar]` satırları ve sütunları aynı anda seçer; burada kadınlar (`sex == 0`) ve üç sütun seçilmektedir.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
print(heart.isna().sum()[heart.isna().sum() > 0])
filled = heart["ca"].fillna(heart["ca"].median())
print("Doldurmadan sonra ca sütununda kalan eksik değer:", filled.isna().sum())
print("Hiç eksik değeri olmayan satır sayısı:", len(heart.dropna()))

**Python notu.** Eksik bir değer `NaN` (not a number, sayı değil) olarak görünür. `isna()` eksik değerleri işaretler, `.sum()` ise her sütundaki işaretleri sayar. `fillna(değer)` eksik değerlerin yerine bir değer koyar (burada sütunun ortancası kullanılmıştır); `dropna()` ise eksik değeri olan her satırı siler. Bu haftanın modelindeki değişkenlerde eksik değer yoktur; bu yüzden defter 303 hastanın tamamını kullanır.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
pd.get_dummies(heart["cp"], prefix="cp", drop_first=True, dtype=int).head()

**Python notu.** `cp` sütunu göğüs ağrısının tipini kodlar: 1 tipik anjina, 2 atipik anjina, 3 anjina dışı ağrı, 4 asemptomatik. Buradaki sayılar bir miktarı değil, bir kategoriyi gösterir: 4. tip, 2. tipin iki katı değildir. Bu yüzden `pd.get_dummies` her tip için 0 ya da 1 değerini alan bir *kukla değişken* oluşturur. `drop_first=True` 1. tipi dışarıda bırakır ve onu referans yapar: Üç sütunun hepsinde 0 olan bir hastada tipik anjina vardır.

In [ ]:
# Modelin tablosu: Her satır bir hasta, bütün değerler sayı
FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
data = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"],
                      drop_first=True, dtype=int)
X = data.drop(columns="disease")
y = data["disease"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
print("X:", X.shape, " eğitim kümesi:", X_train.shape, " test kümesi:", X_test.shape)
print("Hastalık oranı: eğitim", round(y_train.mean(), 3),
      " test", round(y_test.mean(), 3))

**Python notu.** `train_test_split` dört nesne döndürür; `X_train, X_test, y_train, y_test = ...` satırı bunları bu sırayla dört ada dağıtır. `test_size=0.25` hastaların dörtte birini ayrı tutar, `stratify=y` hastalığı olanların oranını iki parçada da aynı tutar, `random_state` ise ayırmanın her çalıştırmada aynı olmasını sağlar. Burada her hastanın bir satırı vardır. Bir hastanın birden fazla satırı olduğunda, örneğin birden fazla EKG'si bulunduğunda, ayırma bir hastanın bütün satırlarını aynı tarafta tutmalıdır; `GroupShuffleSplit` bunu hasta kimliğiyle yapar.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
first = LogisticRegression(max_iter=1000)
first.fit(X_train, y_train)
print(first.predict(X_test[:5]))
print(first.predict_proba(X_test[:5]).round(3))

**Python notu.** scikit-learn'deki her model aynı kalıbı izler. `fit(X, y)` eğitim verisinden öğrenir. `predict(X)` 0 ya da 1 olan bir etiket döndürür; `predict_proba(X)` ise her sınıf için bir olasılık sütunu verir: İlk sütun 0'ın, ikinci sütun 1'in olasılığıdır. `[:5]` ilk beş satırı alır; `max_iter=1000` ise modelin eğitiminin yeterince uzun sürmesine izin verir.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)
p_test = model.predict_proba(X_test)[:, 1]
print("İlk beş test hastasının olasılıkları:", p_test[:5].round(3))

**Python notu.** `make_pipeline` adımları tek bir modelde art arda bağlar. `StandardScaler` her sütunu eğitim verisinin ortalaması ve standart sapmasıyla bir *z skoruna*, yani $(x - \bar{x})/s$ değerine dönüştürür. İşlem hattının içinde `fit` bu değerleri yalnızca eğitim hastalarından öğrenir ve test hastaları aynı sayılarla ölçeklenir; böylece test kümesinden modele hiçbir bilgi sızmaz. `[:, 1]` iki boyutlu bir dizinin bütün satırlarını (`:`) ve ikinci sütununu (`1`) alır.

**Hızlı kontrol.** Bir model `(76, 2)` boyutunda bir `predict_proba(X_test)` döndürüyor. 76 ve 2 neyi anlatır?

<details><summary>Cevap</summary>

76 satır, test kümesindeki 76 hastadır. 2 sütun 0 ve 1 sınıflarının olasılıklarıdır ve her satırda toplamları 1'dir.

</details>

**Alıştırma 2.1.** `heart` tablosunun bütün satır ve sütunlarında toplam kaç hücre eksiktir?

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma 2.1", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = int(heart.isna().sum().sum())
teach.check("Alıştırma 2.1", answer)

**Alıştırma 2.2.** 303 hasta `test_size=0.25` ile ayrıldığında test kümesine kaç hasta düşer? Cevabı `len(...)` ile verin.

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma 2.2", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = len(X_test)
teach.check("Alıştırma 2.2", answer)

**Alıştırma 2.3.** `model` işlem hattının test kümesindeki doğruluğu, yani doğru etiketlenen hastaların oranı nedir? `model.score(X_test, y_test)` kullanın ve sonucu üç ondalık basamağa yuvarlayın.

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma 2.3", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = round(model.score(X_test, y_test), 3)
teach.check("Alıştırma 2.3", answer)

## 1. İlk model: Koroner arter hastalığı için lojistik regresyon

*Ders: [Lojistik regresyon: Risk faktörlerinden olasılığa](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html#logistic-regression-from-risk-factors-to-a-probability)*

Oturumun mini projesi, 303 Cleveland hastasının yedi değişkeninden ve göğüs ağrısı tipinden koroner arter hastalığı (KAH) olasılığını kestiren bir model kurar. Model eğitim kümesinde kurulur ve bu bölümde incelenir; 2. bölüm ise modelin başarımını test kümesinde ölçer.

**Formüllerden koda.** Doğrusal öngörücü $z = \beta_0 + \sum_j \beta_j x_j$, $\beta_0$ için `intercept_`, $\beta_j$ için `coef_` değerlerini kullanır. $p = 1/(1 + e^{-z})$ olasılığı, `predict_proba` çıktısının ikinci sütunudur. Bir değişkenin odds oranı $e^{\beta_j}$, yani `np.exp(coef_)` değeridir. `StandardScaler` sonrasında $x_j$ değerleri z skorudur; bu yüzden her odds oranı, değişkeninin bir standart sapmasına (SS) karşılık gelir.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 1. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB02_makine_ogrenmesi_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(1)

In [ ]:
# Modelin katsayıları: standart sapma başına odds oranları
lr = model.named_steps["logisticregression"]
coefs = pd.DataFrame({"beta": lr.coef_[0], "odds oranı": np.exp(lr.coef_[0])},
                     index=X.columns).sort_values("beta")
print("Sabit terim beta_0 =", round(lr.intercept_[0], 3))
coefs.round(3)

**Python notu.** `model.named_steps` bir işlem hattının adımlarını adlarıyla tutar; dolayısıyla `named_steps["logisticregression"]`, eğitilmiş lojistik regresyon modelidir. `coef_[0]` modelin ilk ve tek katsayı satırıdır. `sort_values("beta")` tabloyu en koruyucu değişkenden riski en çok artıran değişkene doğru sıralar.

In [ ]:
# Haftanın vakası koşu bandında: z ve p elle ve modelle
case = pd.DataFrame([{"age": 57, "sex": 1, "trestbps": 119, "chol": 230, "thalach": 132,
                      "exang": 1, "oldpeak": 1.6, "cp_2.0": 1, "cp_3.0": 0, "cp_4.0": 0}])[X.columns]
scaler = model.named_steps["standardscaler"]
zscores = (case.to_numpy()[0] - scaler.mean_) / scaler.scale_
z = lr.intercept_[0] + np.sum(lr.coef_[0] * zscores)
p_hand = 1 / (1 + np.exp(-z))
print("z =", round(z, 3), " p (elle) =", round(p_hand, 3),
      " p (model) =", round(model.predict_proba(case)[0, 1], 3))

**Python notu.** `scaler.mean_` ve `scaler.scale_`, ölçekleyicinin eğitim kümesinden öğrendiği ortalama ve standart sapmadır. `case.to_numpy()[0]` tek satırlık tabloyu on sayılık bir diziye çevirir. `np.sum(a * b)` iki diziyi eleman eleman çarpar ve çarpımları toplar; bu da formüldeki $\sum_j \beta_j x_j$ toplamıdır.

**Alıştırma A.** VC-0246 numaralı hastada atipik anjina vardır (`cp_2.0 = 1`). Ağrısı tipik anjina olsaydı, yani üç `cp` sütununun hepsi 0 olsaydı, `model` işlem hattına göre KAH olasılığı ne olurdu? Sonucu üç ondalık basamağa yuvarlayın.

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma A", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
typical = case.copy()
typical["cp_2.0"] = 0
answer = round(model.predict_proba(typical)[0, 1], 3)
teach.check("Alıştırma A", answer)

**Gözlemle ve düşün.** Cleveland verisinde göğüs ağrısı olmayan hastalarda (`cp_4.0 = 1`) hastalığın odds değeri en yüksektir; referans olan tipik anjinada ise daha düşüktür. Hastaların sevk edilme biçimi bunu nasıl açıklayabilir? Bu durum, modelin farklı bir hasta karışımı gören bir klinikte kullanılması için ne anlama gelir?

**İleri düzey (isteğe bağlı).** Aynı işlem hattını `LogisticRegression(C=0.05, max_iter=1000)` ile yeniden eğitin. Küçük bir `C` güçlü bir *düzenlileştirme* anlamına gelir: Katsayılar sıfıra doğru küçülür. İki odds oranı tablosunu ve 2. bölümdeki test AUC değerini karşılaştırın.

In [ ]:
# İleri düzey (isteğe bağlı)
strong = make_pipeline(StandardScaler(), LogisticRegression(C=0.05, max_iter=1000)).fit(X_train, y_train)
pd.DataFrame({"C=1": np.exp(lr.coef_[0]),
              "C=0.05": np.exp(strong.named_steps["logisticregression"].coef_[0])},
             index=X.columns).round(2)

## 2. Bir modeli değerlendirmek: Karışıklık matrisi ve ROC eğrisi

*Ders: [Bir modeli değerlendirmek: Karışıklık matrisi ve ROC eğrisi](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html#measuring-a-model-confusion-matrix-and-roc-curve)*

Test kümesinde modelin hiç görmediği 76 hasta vardır. Bir eşik her olasılığı bir karara dönüştürür; kararlar gerçek durumla karşılaştırılarak bir *karışıklık matrisinde* sayılır. Eşiği kaydırmak duyarlılığı artırırken özgüllüğü azaltır ya da tersini yapar; *alıcı işletim karakteristiği (ROC) eğrisi* ise bütün eşikleri aynı anda gösterir.

**Formüllerden koda.** `tp`, `fp`, `tn` ve `fn` sayılarıyla duyarlılık $Se = TP/(TP+FN)$, özgüllük $Sp = TN/(TN+FP)$, pozitif öngörü değeri $PPV = TP/(TP+FP)$, negatif öngörü değeri ise $NPV = TN/(TN+FN)$ olur. `roc_curve` yanlış pozitif oranını ($1 - Sp$), doğru pozitif oranını ($Se$) ve eşikleri döndürür; `roc_auc_score` ise eğri altındaki alanı (AUC) verir. Youden indeksi $J = Se + Sp - 1$ olarak tanımlanır.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 2. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB02_makine_ogrenmesi_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(2)

In [ ]:
# 0,5 eşiğinde karışıklık matrisi
pred = (p_test >= 0.5).astype(int)
print(pd.crosstab(y_test.to_numpy(), pred, rownames=["gerçek"], colnames=["model"]))
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
print(f"TP={tp} FP={fp} TN={tn} FN={fn}")
print("Se =", round(tp / (tp + fn), 3), " Sp =", round(tn / (tn + fp), 3),
      " PPV =", round(tp / (tp + fp), 3), " NPV =", round(tn / (tn + fn), 3))

**Python notu.** `(p_test >= 0.5)` her hasta için `True` ya da `False` verir; `.astype(int)` bunları 1 ve 0'a çevirir. `pd.crosstab` gerçek durum ve karar çiftlerini sayar. `confusion_matrix(...).ravel()` dört sayıyı TN, FP, FN, TP sırasıyla döndürür; aynı satır bunları dört ada dağıtır. Bir *f-dizgesi* süslü parantez içindeki değerleri metnin içine yerleştirir.

In [ ]:
# ROC eğrisi, AUC ve Youden indeksi
fpr, tpr, thresholds = roc_curve(y_test, p_test)
auc = roc_auc_score(y_test, p_test)
best = np.argmax(tpr - fpr)
print("AUC =", round(auc, 3))
print("Youden: eşik", round(thresholds[best], 3), " Se", round(tpr[best], 3),
      " Sp", round(1 - fpr[best], 3), " J", round(tpr[best] - fpr[best], 3))
fig, ax = plt.subplots(figsize=(4.8, 4.6))
ax.plot(fpr, tpr, color="#9B1B1B", lw=2, label=f"lojistik regresyon (AUC {auc:.3f})")
ax.plot([0, 1], [0, 1], color="#888888", ls="--", lw=1, label="şans")
ax.scatter(fpr[best], tpr[best], color="#1F5F8B", zorder=3, label="Youden")
ax.set_xlabel("1 − özgüllük (yanlış pozitif oranı)")
ax.set_ylabel("Duyarlılık (doğru pozitif oranı)")
ax.legend(loc="lower right")
plt.show()

**Python notu.** `roc_curve` anlamlı bütün eşikleri dener ve aynı uzunlukta üç dizi döndürür. `np.argmax(tpr - fpr)`, $Se - (1 - Sp) = J$ değerinin en büyük olduğu konumu verir; bu konum da karşılık gelen eşiği seçer.

**Keşfet.** Eşiği kaydırın. Karışıklık matrisi, dört ölçüt ve ROC eğrisi üzerindeki nokta birlikte değişir. Düşük bir eşik kimseyi kaçırmaması gereken bir teste, yüksek bir eşik ise girişimsel bir işlemi ancak yeterince emin olduğunda başlatması gereken bir teste uyar.

In [ ]:
# Keşfet: eşik
def show_threshold(threshold=0.5):
    pred = (p_test >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    se, sp = tp / (tp + fn), tn / (tn + fp)
    ppv = tp / (tp + fp) if tp + fp else float("nan")
    fig, ax = plt.subplots(figsize=(4.4, 4.2))
    ax.plot(fpr, tpr, color="#9B1B1B", lw=2)
    ax.plot([0, 1], [0, 1], color="#888888", ls="--", lw=1)
    ax.scatter(1 - sp, se, s=60, color="#1F5F8B", zorder=3)
    ax.set_xlabel("1 − Sp")
    ax.set_ylabel("Se")
    ax.set_title(f"eşik {threshold:.2f}")
    plt.show()
    print(f"TP={tp} FP={fp} TN={tn} FN={fn} | Se={se:.3f} Sp={sp:.3f} PPV={ppv:.3f}")


try:
    from ipywidgets import interact, FloatSlider
    interact(show_threshold, threshold=FloatSlider(value=0.5, min=0.05, max=0.95, step=0.05,
                                                   description="eşik"))
except ImportError:
    for value in (0.3, 0.5, 0.7):
        show_threshold(value)

**Alıştırma B.** `model` işlem hattının test kümesindeki AUC değeri nedir? Sonucu üç ondalık basamağa yuvarlayın.

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma B", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = round(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]), 3)
teach.check("Alıştırma B", answer)

**Alıştırma C.** `model` işlem hattının test kümesinde 0,5 eşiğindeki duyarlılığı nedir? Sonucu üç ondalık basamağa yuvarlayın.

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma C", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
pred = (model.predict_proba(X_test)[:, 1] >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
answer = round(tp / (tp + fn), 3)
teach.check("Alıştırma C", answer)

**Gözlemle ve düşün.** 0,93'lük bir AUC mükemmel görünür. Bir kardiyolog, modeli anjiyografi kararları için kullanmadan önce hâlâ hangi soruları sormalıdır? Test kümesinin büyüklüğünü, hastaların nereden geldiğini ve bir yanlış negatifin bedelini bir yanlış pozitifinkiyle karşılaştırarak düşünün.

## 3. Çapraz doğrulama ve veri sızıntısı

*Ders: [Eğitim ve test: Çapraz doğrulama ve veri sızıntısı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html#training-and-testing-cross-validation-and-data-leakage)*

76 hastalık tek bir test kümesi tek bir sayı verir; başka bir ayırma başka bir sayı verirdi. *K katlı çapraz doğrulama* hastaları $K$ parçaya böler, $K-1$ parçada eğitir, kalan parçada test eder ve bunu $K$ kez tekrarlar. $K$ sonucun ortalaması daha kararlı bir kestirimdir; sonuçların yayılımı ise bu kestirimin ne kadar oynadığını gösterir. Veriden öğrenen her adım (ölçekleme ve değişken seçimi dâhil) her katın içinde yeniden yapılmalıdır; aksi hâlde test hastalarının bilgisi modele *sızar*.

**Formüllerden koda.** Kestirim $\overline{AUC} = \frac{1}{K}\sum_{k=1}^{K} AUC_k$ olarak hesaplanır. `StratifiedKFold(5, shuffle=True, random_state=...)` her birinde hastalık oranı aynı olan beş kat oluşturur; `cross_val_score` ise beş $AUC_k$ değerini döndürür.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 3. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB02_makine_ogrenmesi_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(3)

In [ ]:
# Bütün işlem hattının beş katlı çapraz doğrulaması
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
scores = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc")
print("Beş katın AUC değerleri:", scores.round(3))
print("Ortalama", round(scores.mean(), 3), " SS", round(scores.std(), 3))

**Python notu.** `cross_val_score` henüz eğitilmemiş işlem hattını ve `X` ile `y` verilerinin tamamını alır. Her katta işlem hattının yeni bir kopyasını dört parçada eğitir; böylece ölçekleyici de yalnızca eğitim parçalarından öğrenir. Ardından beşinci parçayı puanlar. `scoring="roc_auc"` AUC değerinin hesaplanmasını ister.

In [ ]:
# Saf gürültüde sızıntı: 500 rastgele değişken ve hiçbir gerçek sinyal
rng = np.random.default_rng(RANDOM_STATE)
noise = rng.normal(size=(len(y), 500))

# Yanlış: 10 "en iyi" değişkeni bütün hastalarda seç, sonra çapraz doğrula
chosen = SelectKBest(f_classif, k=10).fit_transform(noise, y)
leaky = cross_val_score(LogisticRegression(max_iter=1000), chosen, y, cv=cv, scoring="roc_auc")

# Doğru: Seçim işlem hattının bir adımıdır ve her katta tekrarlanır
honest = cross_val_score(make_pipeline(SelectKBest(f_classif, k=10), LogisticRegression(max_iter=1000)),
                         noise, y, cv=cv, scoring="roc_auc")
print("Çapraz doğrulamadan önce seçim: AUC", round(leaky.mean(), 3))
print("İşlem hattı içinde seçim:          AUC", round(honest.mean(), 3))

**Python notu.** `rng.normal(size=(303, 500))` 303 satır ve 500 sütundan oluşan bir rastgele sayı tablosu üretir. `SelectKBest(f_classif, k=10)` iki grup arasında en çok farklılaşan 10 sütunu tutar. Saf gürültüde bazı sütunlar şans eseri farklılaşır; bu sütunları bütün hastalarla seçmek, test katlarını da seçime ortak eder.

**Alıştırma D.** `pipe` işlem hattının `cv` katları üzerindeki ortalama çapraz doğrulama AUC değeri nedir? Sonucu üç ondalık basamağa yuvarlayın.

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma D", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = round(cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc").mean(), 3)
teach.check("Alıştırma D", answer)

**Gözlemle ve düşün.** Sızıntılı sürüm, hiçbir bilgi içermeyen veride yaklaşık 0,7'lik bir AUC bulmaktadır. Yayımlanmış modeller de bazen bu şekilde kurulur. Bir makalenin yöntem bölümündeki hangi cümle bu konuda sizi şüphelendirir, hangisi güven verir?

**İleri düzey (isteğe bağlı).** Çapraz doğrulamayı `sklearn.model_selection` içindeki `RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)` ile farklı karıştırmalarla on kez tekrarlayın ve 50 AUC değerinin histogramını çizin.

In [ ]:
# İleri düzey (isteğe bağlı)
from sklearn.model_selection import RepeatedStratifiedKFold
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)
many = cross_val_score(pipe, X, y, cv=rcv, scoring="roc_auc")
plt.hist(many, bins=12, color="#1F5F8B")
plt.xlabel("AUC")
plt.ylabel("Kat sayısı")
plt.show()
print(round(many.mean(), 3), round(many.std(), 3))

## 4. Karar ağaçları ve rastgele ormanlar

*Ders: [Karar ağaçları ve rastgele ormanlar](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html#decision-trees-and-random-forests)*

Bir karar ağacı, klinik bir algoritma gibi her seferinde tek bir evet-hayır sorusu sorar. Her düğümde iki grubu en saf hâle getiren soruyu seçer. Rastgele orman ise yüzlerce ağaç kurar ve bu ağaçların verdiği olasılıkların ortalamasını alır; her ağaç hastaların bir bootstrap örneğinde kurulur ve her düğümde değişkenlerin rastgele bir alt kümesine bakar.

**Formüllerden koda.** Sınıf oranları $p_0$ ve $p_1$ olan bir düğümün Gini safsızlığı $G = 1 - (p_0^2 + p_1^2)$ olarak tanımlanır. Bir bölünme, iki alt düğümün büyüklükleriyle ağırlıklandırılmış safsızlığıyla puanlanır. `DecisionTreeClassifier(max_depth=3)` ağacı üç soru düzeyiyle sınırlar; `RandomForestClassifier(n_estimators=300)` ise 300 ağacın ortalamasını alır.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 4. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB02_makine_ogrenmesi_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(4)

In [ ]:
# Üç soru düzeyli bir ağaç
tree = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE).fit(X_train, y_train)
print("Ağacın test AUC değeri:", round(roc_auc_score(y_test, tree.predict_proba(X_test)[:, 1]), 3))
fig, ax = plt.subplots(figsize=(13, 5.5))
plot_tree(tree, feature_names=list(X.columns), class_names=["KAH yok", "KAH"],
          filled=True, impurity=True, fontsize=8, ax=ax)
plt.show()

**Python notu.** `plot_tree` her düğümü sorusu, Gini safsızlığı, hasta sayısı (`samples`) ve her sınıfın sayısıyla (`value`) çizer. Bir düğümün sol dalı, sorunun doğru olduğu hastaları içerir.

**Alıştırma E.** Ağacın kökündeki, yani hiçbir soru sorulmadan önceki Gini safsızlığını `y_train` içindeki iki sınıfın sayılarından hesaplayın. Sonucu üç ondalık basamağa yuvarlayın.

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma E", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
n0, n1 = (y_train == 0).sum(), (y_train == 1).sum()
p0, p1 = n0 / (n0 + n1), n1 / (n0 + n1)
answer = round(1 - (p0 ** 2 + p1 ** 2), 3)
teach.check("Alıştırma E", answer)

In [ ]:
# Rastgele orman ve değişkenlerinin permütasyon önemi
forest = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X_train, y_train)
print("Ormanın test AUC değeri:", round(roc_auc_score(y_test, forest.predict_proba(X_test)[:, 1]), 3))
imp = permutation_importance(forest, X_test, y_test, scoring="roc_auc", n_repeats=20,
                             random_state=RANDOM_STATE)
order = np.argsort(imp.importances_mean)
fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(X.columns[order], imp.importances_mean[order], xerr=imp.importances_std[order], color="#1F5F8B")
ax.set_xlabel("Değişken karıştırıldığında AUC'deki düşüş")
plt.show()

**Python notu.** *Permütasyon önemi*, test kümesindeki sütunları tek tek karıştırır ve AUC'nin ne kadar düştüğünü ölçer. Büyük bir düşüş, ormanın o değişkene dayandığını gösterir. `np.argsort` değerleri sıralayacak konumları döndürür; böylece çubuklar en küçükten en büyüğe doğru dizilir.

**Gözlemle ve düşün.** Tek bir ağaç kolay okunur ama ormandan daha düşük bir AUC'ye ulaşır; orman ise isabetlidir ama zor okunur. Yatak başındaki bir karar için hangisini tercih edersiniz ve ormana güvenmeden önce neyi görmeniz gerekir?

## 5. Risk polikliniği: Kayıttaki koroner arter hastalığı

*Ders: [Sanal klinik, 2. aşama: Risk polikliniği açılıyor](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html#the-virtual-clinic-stage-2-the-risk-outpatient-clinic-opens)*

Sanal klinik 2.000 hastaya ulaşır ve risk polikliniğini açar. Ders, her hastanın koroner arter hastalığını `vclinic.TRUE_EFFECTS` içinde saklanan bilinen log-odds etkilerinden üretmiştir. Bu nedenle kayıtta kurulan bir lojistik regresyonun bu değerlere yakın katsayılar bulması beklenir; gerçek veride kurulan bir modelde ise böyle bir karşılaştırma hiçbir zaman yapılamaz. `vclinic.add_followup`, 6. bölümden itibaren kullanılan ekokardiyografi ayrıntılarını ve izlem verilerini ekler.

**Formüllerden koda.** Değişkenler `TRUE_EFFECTS` ile aynı biçimde kodlanır: 60 yaşın üstündeki her 10 yıl, $(LDL - 130)/40$, $(HDL - 50)/15$ ve diğerleri için 0 ya da 1. `LogisticRegression(C=np.inf)` modeli düzenlileştirme olmadan eğitir; bu yüzden `coef_` değerleri $\beta_j$ katsayılarının en çok olabilirlik kestirimleridir.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 5. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB02_makine_ogrenmesi_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(5)

In [ ]:
# İzlem verileriyle kayıt: 2.000 hasta
registry = vclinic.add_followup(vclinic.make_cohort(2000, seed=RANDOM_STATE))
print(registry.shape)
print(registry["main_diagnosis"].value_counts().to_string())


def risk_table(df):
    """Değişkenler vclinic.TRUE_EFFECTS ile aynı biçimde kodlanır."""
    return pd.DataFrame({
        "age_per_10y": (df["age"] - 60) / 10,
        "male": (df["sex"] == "M").astype(int),
        "smoker": df["smoker"],
        "diabetes": df["diabetes"],
        "hypertension": df["hypertension"],
        "ldl_per_40": (df["ldl"] - 130) / 40,
        "hdl_per_15": (df["hdl"] - 50) / 15,
        "family_history": df["family_history"],
    })


Z = risk_table(registry)
cad_model = LogisticRegression(C=np.inf, max_iter=2000).fit(Z, registry["cad"])
truth = pd.Series(vclinic.TRUE_EFFECTS)
compare = pd.DataFrame({"öğrenilen": np.r_[cad_model.intercept_, cad_model.coef_[0]],
                        "gerçek": truth[["intercept"] + list(Z.columns)].to_numpy()},
                       index=["intercept"] + list(Z.columns))
compare.round(2)

**Python notu.** `def` satırının hemen altında üç tırnak içindeki metin, yani *docstring*, fonksiyonun ne yaptığını anlatır. `np.r_[a, b]` iki diziyi tek bir dizide birleştirir. `truth[["intercept"] + list(Z.columns)]` gerçek değerleri öğrenilenlerle aynı sıraya koyar; böylece tablonun iki sütunu satır satır karşılaştırılabilir.

In [ ]:
# Haftanın vakası VC-0246 risk polikliniğinde
row = registry[registry["patient_id"] == "VC-0246"]
z_true = truth["intercept"] + (risk_table(row).iloc[0] * truth[Z.columns]).sum()
print(row[["age", "sex", "smoker", "hypertension", "diabetes", "ldl", "hdl", "family_history"]].to_string(index=False))
print("KAH olasılığı, model:", round(cad_model.predict_proba(risk_table(row))[0, 1], 3),
      " gerçek:", round(1 / (1 + np.exp(-z_true)), 3))
print("Kayıttaki koroner anjiyografi (cad):", int(row["cad"].iloc[0]))

**Alıştırma F.** `cad_model`, `smoker` değişkeni için hangi log-odds katsayısını öğrenmiştir? Sonucu iki ondalık basamağa yuvarlayın ve `TRUE_EFFECTS["smoker"]` ile karşılaştırın.

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma F", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = round(cad_model.coef_[0][list(Z.columns).index("smoker")], 2)
teach.check("Alıştırma F", answer)

**Gözlemle ve düşün.** Model doğru biçimde kurulmuş olduğu hâlde öğrenilen katsayılar gerçek değerlere yakındır ama onlara eşit değildir. 200 hastayla ve 20.000 hastayla bu farklar nasıl değişirdi? VC-0246'nın olasılığı yaklaşık 0,8'dir ve anjiyogramı normaldir. Model bu hasta için yanılmış mıdır?

## 6. Nadir sonuçlar: Dengesiz veri

*Ders: [Nadir sonuçlar: Dengesiz veri](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html#rare-outcomes-imbalanced-data)*

12 ay içinde ölüm ya da kalp yetersizliği nedeniyle hastaneye yatış, kayıttaki her 100 hastanın yaklaşık 3'ünde görülür. Bu kadar nadir bir sonuçta her zaman *olay yok* diyen bir model 100 hastanın 97'sinde haklı çıkar ama hiçbir işe yaramaz. Bu bölüm doğruluğun neden yanılttığını, sınıf ağırlıklarının modeli nadir sınıfa doğru nasıl kaydırdığını ve pozitif öngörü değerinin neden prevalansa bağlı olduğunu gösterir.

**Formüllerden koda.** Bayes teoremi, $\pi$ prevalansı için $PPV = \frac{Se \cdot \pi}{Se \cdot \pi + (1 - Sp)(1 - \pi)}$ sonucunu verir. `class_weight="balanced"` her sınıfa $n/(2 n_k)$ ağırlığını verir; böylece nadir sınıf, sık görülen sınıf kadar hesaba katılır. `precision_recall_curve` her eşikte PPV'yi (*kesinlik*) ve duyarlılığı (*recall*) döndürür; `average_precision_score` ise eğriyi tek sayıyla özetler.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 6. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB02_makine_ogrenmesi_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(6)

In [ ]:
# Nadir sonuç: 12 ay içinde olay
registry["event_12m"] = ((registry["event"] == 1) & (registry["followup_months"] <= 12)).astype(int)
EVENT_FEATURES = ["age", "egfr", "lvef", "ntprobnp", "af", "diabetes"]
Xe = registry[EVENT_FEATURES].assign(ntprobnp=np.log(registry["ntprobnp"]))
ye = registry["event_12m"]
Xe_train, Xe_test, ye_train, ye_test = train_test_split(
    Xe, ye, test_size=0.3, stratify=ye, random_state=RANDOM_STATE)
print("Prevalans:", round(ye.mean(), 3), " test kümesindeki olay:", int(ye_test.sum()), "/", len(ye_test))
print("Her zaman olay yok demenin doğruluğu:", round((ye_test == 0).mean(), 3))

rows = []
for weight in [None, "balanced"]:
    m = make_pipeline(StandardScaler(), LogisticRegression(class_weight=weight, max_iter=1000)).fit(Xe_train, ye_train)
    p = m.predict_proba(Xe_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(ye_test, (p >= 0.5).astype(int)).ravel()
    rows.append({"class_weight": str(weight), "AUC": roc_auc_score(ye_test, p),
                 "AP": average_precision_score(ye_test, p), "doğruluk": (tp + tn) / len(ye_test),
                 "Se": tp / (tp + fn), "PPV": tp / (tp + fp) if tp + fp else np.nan})
pd.DataFrame(rows).round(3)

**Python notu.** `.assign(ntprobnp=...)` bir sütunu değiştirilmiş bir tablo kopyası döndürür; burada NT-proBNP'nin logaritması alınmıştır, çünkü NT-proBNP değerleri birkaç büyüklük basamağına yayılır. `str(weight)` tablo için `None` değerini "None" metnine çevirir. `np.nan` hesaplanamayan bir değeri işaretler; burada hiç pozitif karar verilmediği için PPV hesaplanamamıştır.

In [ ]:
# Kesinlik ve duyarlılık; prevalansa göre PPV
balanced = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)).fit(Xe_train, ye_train)
precision, recall, _ = precision_recall_curve(ye_test, balanced.predict_proba(Xe_test)[:, 1])
prev = np.linspace(0.005, 0.5, 200)
se, sp = 0.85, 0.90
ppv = se * prev / (se * prev + (1 - sp) * (1 - prev))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
ax[0].plot(recall, precision, color="#9B1B1B")
ax[0].axhline(ye_test.mean(), color="#888888", ls="--", label="prevalans")
ax[0].set_xlabel("Duyarlılık (recall)")
ax[0].set_ylabel("Kesinlik (PPV)")
ax[0].legend()
ax[1].plot(prev * 100, ppv, color="#1F5F8B")
ax[1].set_xlabel("Prevalans (%)")
ax[1].set_ylabel("PPV")
ax[1].set_title("Se 0,85, Sp 0,90")
plt.show()

**Python notu.** `np.linspace(0.005, 0.5, 200)` eşit aralıklı 200 prevalans değeri üretir. Bayes teoreminin formülü, NumPy dizilerinde eleman eleman çalıştığı için bu değerlerin hepsine aynı anda uygulanır. `_`, kullanılmayacak bir dönüş değerine verilen addır; burada eşikler kullanılmamaktadır.

**Alıştırma G.** Bir tarama modelinin duyarlılığı 0,85, özgüllüğü 0,90'dır. Prevalansın yüzde 5 olduğu bir toplulukta PPV nedir? Sonucu üç ondalık basamağa yuvarlayın.

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma G", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
se, sp, prev = 0.85, 0.90, 0.05
answer = round(se * prev / (se * prev + (1 - sp) * (1 - prev)), 3)
teach.check("Alıştırma G", answer)

**Gözlemle ve düşün.** Sınıf ağırlıklarıyla model olayların yaklaşık üçte ikisini bulur, ama alarmlarının ancak yaklaşık onda biri gerçek bir olaydır. Böyle bir model, bir kalp yetersizliği polikliniğinin kontrol vizitlerini planlamak için yararlı olur mu? Hangi eşiği seçerdiniz ve bu seçime kimler katılmalıdır?

## 7. Alt grupları bulmak: Kalp yetersizliği olan hastalarda k-ortalamalar

*Ders: [Denetimsiz öğrenme: k-ortalamalar ve fenogruplar](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html#unsupervised-learning-k-means-and-phenogroups)*

Kayıtta düşük ejeksiyon fraksiyonlu (HFrEF) ya da korunmuş ejeksiyon fraksiyonlu (HFpEF) kalp yetersizliği olan yaklaşık 400 hasta bulunur. Denetimsiz öğrenme, hiçbir sonuç bilgisi olmadan gruplar arar. *k-ortalamalar* algoritması $k$ merkez yerleştirir, her hastayı en yakın merkeze atar, her merkezi kendi hastalarının ortalamasına taşır ve hiçbir şey değişmeyene kadar bunu tekrarlar. Ders, kalp yetersizliği olan her hastaya gizli bir fenogrup atamıştır. Kümeleme bu bilgiyi görmez; bölümün sonunda kümeler bu gizli fenogruplarla karşılaştırılır.

**Formüllerden koda.** k-ortalamalar küme içi kareler toplamını, $WCSS = \sum_{k}\sum_{i \in C_k} \lVert \mathbf{x}_i - \mu_k \rVert^2$, en küçük yapar; `KMeans` bu değeri `inertia_` olarak bildirir. Bir hastanın siluet değeri $s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$ olarak hesaplanır; burada $a(i)$ kendi kümesine, $b(i)$ ise en yakın diğer kümeye olan ortalama uzaklıktır. `silhouette_score` bütün hastaların ortalamasını döndürür.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 7. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB02_makine_ogrenmesi_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(7)

In [ ]:
# Kalp yetersizliği olan hastaların standartlaştırılmış öznitelikleri
hf = registry[registry["main_diagnosis"].isin(["HFrEF", "HFpEF"])].copy()
HF_FEATURES = ["age", "bmi", "egfr", "lvef", "ntprobnp", "lavi", "e_e_prime", "lv_mass_index", "tapse_mm"]
X_hf = StandardScaler().fit_transform(hf[HF_FEATURES].assign(ntprobnp=np.log(hf["ntprobnp"])))
print(len(hf), "kalp yetersizliği hastası")

ks = range(2, 7)
wcss, sil = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_hf)
    wcss.append(km.inertia_)
    sil.append(silhouette_score(X_hf, km.labels_))
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
ax[0].plot(list(ks), wcss, "o-", color="#1F5F8B")
ax[0].set_xlabel("k")
ax[0].set_ylabel("WCSS")
ax[1].plot(list(ks), sil, "o-", color="#9B1B1B")
ax[1].set_xlabel("k")
ax[1].set_ylabel("Ortalama siluet")
plt.show()
print(pd.DataFrame({"k": list(ks), "WCSS": np.round(wcss, 1), "siluet": np.round(sil, 3)}).to_string(index=False))

**Python notu.** `.isin([...])` değeri listedeki değerlerden biri olan satırları tutar. `range(2, 7)` 2'den 6'ya kadar olan sayıları verir; döngü her biri için bir k-ortalamalar modeli kurar. `n_init=10` her modeli merkezlerin on farklı rastgele konumundan başlatır ve en iyi sonucu tutar; çünkü k-ortalamalar kötü bir çözümde takılıp kalabilir.

In [ ]:
# Üç küme, ilk iki temel bileşen üzerinde
km3 = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE).fit(X_hf)
hf["cluster"] = km3.labels_
pca = PCA(n_components=2).fit(X_hf)
pcs = pca.transform(X_hf)
fig, ax = plt.subplots(figsize=(5.6, 4.6))
for c, color in zip(range(3), ["#9B1B1B", "#1F5F8B", "#B97813"]):
    ax.scatter(pcs[hf["cluster"] == c, 0], pcs[hf["cluster"] == c, 1], s=14, color=color, label=f"küme {c}")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.0f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.0f}%)")
ax.legend()
plt.show()
hf.groupby("cluster")[HF_FEATURES + ["diabetes", "af"]].mean().round(1).T

**Python notu.** Temel bileşen analizi (PCA), hastaların en çok farklılaştığı yönleri bulur. `explained_variance_ratio_` her yönün toplam varyanstaki payını verir. Grafik kümeleri yalnızca gösterir, onları oluşturmaz: k-ortalamalar dokuz değişkenin tamamıyla çalışmıştır.

**Alıştırma H.** `X_hf` üzerinde üç kümeli k-ortalamalar çözümünün ortalama siluet değeri nedir? `silhouette_score(X_hf, km3.labels_)` kullanın ve sonucu üç ondalık basamağa yuvarlayın.

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma H", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = round(silhouette_score(X_hf, km3.labels_), 3)
teach.check("Alıştırma H", answer)

In [ ]:
# Kliniğin gizli gerçeği: fenogruplar
print(pd.crosstab(hf["cluster"], hf["hf_phenogroup"]))
print("Düzeltilmiş Rand indeksi:", round(adjusted_rand_score(hf["hf_phenogroup"], hf["cluster"]), 3))

**Gözlemle ve düşün.** Ortalama siluet değeri iki kümede üç kümeye göre biraz daha yüksektir; yine de üç küme gizli fenogruplarla iyi örtüşür. Karşılaştırılacak gizli bir gerçeğin olmadığı gerçek bir çalışmada fenogrup sayısına hangi ölçüt karar vermelidir? Kümeleri tablodaki profillerine göre adlandırın.

## 8. Sağkalım analizi: Kaplan-Meier ve log-rank testi

*Ders: [Sağkalım analizi: Kaplan-Meier ve log-rank testi](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html#survival-analysis-kaplan-meier-and-the-log-rank-test)*

İzlem farklı hastalarda farklı zamanlarda sona erer. Son vizitte hâlâ olaysız olan bir hasta *sansürlenir*: Olay daha sonra gerçekleşebilir, ancak gözlenmemiştir. Kaplan-Meier kestiricisi her hastayı izlendiği süre boyunca hesaba katar. Log-rank testi ise iki ya da daha fazla eğrinin şansın açıklayabileceğinden daha fazla farklılaşıp farklılaşmadığını sorar.

**Formüllerden koda.** Olay görülen her $t_i$ zamanında $n_i$ hasta risk altındadır ve $d_i$ hastada olay olur. Kestirim $S(t) = \prod_{t_i \le t} \left(1 - \frac{d_i}{n_i}\right)$ olarak hesaplanır. Log-rank istatistiği $X^2 = \sum_g \frac{(O_g - E_g)^2}{E_g}$ olarak tanımlanır; burada $O_g$, $g$ grubunda gözlenen olay sayısı, $E_g = \sum_i d_i\, n_{gi}/n_i$ ise beklenen olay sayısıdır. $G$ grup için istatistik, $G - 1$ serbestlik dereceli bir ki-kare dağılımıyla karşılaştırılır.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 8. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB02_makine_ogrenmesi_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(8)

In [ ]:
# Elle Kaplan-Meier
def kaplan_meier(time, event):
    """Basamakların zamanları ve her basamaktan sonraki sağkalım; (0, 1) noktasından başlar."""
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    times, surv, s = [0.0], [1.0], 1.0
    for t in np.unique(time[event == 1]):
        at_risk = np.sum(time >= t)
        events = np.sum((time == t) & (event == 1))
        s = s * (1 - events / at_risk)
        times.append(t)
        surv.append(s)
    return np.array(times), np.array(surv)


def survival_at(times, surv, t):
    """Basamak fonksiyonundan okunan S(t)."""
    return surv[np.searchsorted(times, t, side="right") - 1]


t10 = [2, 3, 3, 5, 6, 8, 9, 11, 12, 12]
e10 = [1, 1, 0, 1, 0, 1, 0, 1, 0, 0]
tt, ss = kaplan_meier(t10, e10)
pd.DataFrame({"ay": tt, "S(t)": ss.round(4)})

**Python notu.** `np.unique` olay görülen farklı zamanları sıralanmış olarak döndürür. `np.sum(time >= t)`, `t` zamanında hâlâ izlenen hastaları, yani $n_i$ değerini sayar; `(time == t) & (event == 1)` koşulu da o zamandaki olayları, yani $d_i$ değerini sayar. `np.searchsorted` bir zamanın eğrinin basamakları arasında nereye düştüğünü bulur.

In [ ]:
# Kümelere ve kalp yetersizliği tipine göre Kaplan-Meier eğrileri
fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.9), sharey=True)
for c, color in zip(range(3), ["#9B1B1B", "#1F5F8B", "#B97813"]):
    d = hf[hf["cluster"] == c]
    tt, ss = kaplan_meier(d["followup_months"], d["event"])
    ax[0].step(tt, ss, where="post", color=color, label=f"küme {c} (n={len(d)})")
for kind, color in zip(["HFrEF", "HFpEF"], ["#9B1B1B", "#1F5F8B"]):
    d = hf[hf["main_diagnosis"] == kind]
    tt, ss = kaplan_meier(d["followup_months"], d["event"])
    ax[1].step(tt, ss, where="post", color=color, label=f"{kind} (n={len(d)})")
    print(kind, "S(24) =", round(survival_at(tt, ss, 24), 3))
for a in ax:
    a.set_xlabel("Ay")
    a.legend(loc="lower left")
ax[0].set_ylabel("Ölüm ve yatış olmadan")
plt.show()


def logrank(time, event, group):
    """İki ya da daha fazla grup için log-rank testi: ki-kare ve p değeri."""
    time, event, group = np.asarray(time, float), np.asarray(event, int), np.asarray(group)
    labels = np.unique(group)
    observed = np.zeros(len(labels))
    expected = np.zeros(len(labels))
    for t in np.unique(time[event == 1]):
        at_risk = time >= t
        d = np.sum((time == t) & (event == 1))
        n = np.sum(at_risk)
        for j, g in enumerate(labels):
            observed[j] += np.sum((time == t) & (event == 1) & (group == g))
            expected[j] += d * np.sum(at_risk & (group == g)) / n
    chi2 = np.sum((observed - expected) ** 2 / expected)
    return chi2, stats.chi2.sf(chi2, len(labels) - 1)


for name, col in [("HFrEF ve HFpEF", "main_diagnosis"), ("kümeler", "cluster")]:
    chi2, p = logrank(hf["followup_months"], hf["event"], hf[col])
    print(f"{name}: X2 = {chi2:.2f}, p = {p:.2g}")

**Python notu.** `ax[0].step(..., where="post")` her değeri bir sonraki zamana kadar koruyan bir basamak fonksiyonu çizer; Kaplan-Meier eğrisi de böyle görünür. SciPy'deki `stats.chi2.sf(x, df)`, en az `x` kadar büyük bir ki-kare değerinin olasılığını, yani p değerini verir. `f"{p:.2g}"` bir sayıyı iki anlamlı basamakla yazdırır.

**Alıştırma I.** HFrEF hastalarında 24. ayda ölüm ya da hastaneye yatış olmadan geçen sürenin Kaplan-Meier kestirimi nedir? Sonucu üç ondalık basamağa yuvarlayın.

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma I", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
d = hf[hf["main_diagnosis"] == "HFrEF"]
tt, ss = kaplan_meier(d["followup_months"], d["event"])
answer = round(survival_at(tt, ss, 24), 3)
teach.check("Alıştırma I", answer)

**Gözlemle ve düşün.** Bu kayıtta HFrEF ile HFpEF arasında belirgin bir fark yokken üç küme birbirinden ayrışmaktadır. Bu, hastaları yalnızca ejeksiyon fraksiyonuna göre gruplamak hakkında ne söyler? Hangi kümeyi daha yakından izlerdiniz ve bakımını değiştirmeden önce başka neyi bilmek isterdiniz?

**İleri düzey (isteğe bağlı).** `lifelines` paketi kuruluysa bu bölümdeki iki fonksiyonu `lifelines` içindeki `KaplanMeierFitter` ve `multivariate_logrank_test` ile karşılaştırın. `lifelines` paketinin log-rank istatistiği tam varyansı kullanır ve bu yüzden biraz daha büyüktür.

In [ ]:
# İleri düzey (isteğe bağlı)
try:
    from lifelines import KaplanMeierFitter
    from lifelines.statistics import multivariate_logrank_test
    d = hf[hf["main_diagnosis"] == "HFrEF"]
    kmf = KaplanMeierFitter().fit(d["followup_months"], d["event"])
    print("lifelines S(24):", round(float(kmf.survival_function_at_times(24).iloc[0]), 3))
    print(multivariate_logrank_test(hf["followup_months"], hf["cluster"], hf["event"]).summary)
except ImportError:
    print("lifelines kurulu değil; bu bölümdeki fonksiyonlar eğitim için yeterlidir.")

## 9. YZ asistanıyla çalışmak, 2. bölüm: Bir modeli denetlemek

*Ders: [YZ asistanıyla çalışmak, 2. bölüm: Bir modeli denetlemek](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html#ai-pair-programming-part-2-reviewing-a-model-built-by-an-assistant)*

Bir katılımcı, bir YZ asistanından *kayıttaki verilerle 12 ay içinde ölümü ya da hastaneye yatışı öngören bir model kurmasını ve modelin AUC değerini vermesini* istemiştir. Aşağıdaki kod asistanın cevabıdır; hatasız çalışır ve mükemmel bir sonuç bildirir. Kodu çalıştırmadan önce dersteki model denetimi sorularıyla okuyun: Model hangi değişkenleri görebilmektedir ve başarımı hangi hastalarda ölçülmektedir?

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 9. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB02_makine_ogrenmesi_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(9)

In [ ]:
# Bir YZ asistanının kodu: 12 aylık olayı öngör ve AUC değerini bildir
features = ["age", "egfr", "lvef", "ntprobnp", "af", "diabetes", "bmi", "sbp", "followup_months"]
rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
rf.fit(registry[features], registry["event_12m"])
auc_reported = roc_auc_score(registry["event_12m"], rf.predict_proba(registry[features])[:, 1])
print("AUC:", round(auc_reported, 3))

**Gözlemle ve düşün.** Kodun iki sorunu vardır. Birincisi, modelin karar anında asla bilemeyeceği bir değişkenle ilgilidir; ikincisi, AUC'nin ölçüldüğü yerle. İki sorunu da adlandırın ve her birini denetim sorularından hangisinin yakaladığını söyleyin.

**Alıştırma J.** Kodu düzeltin: `followup_months` değişkenini çıkarın, `RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=RANDOM_STATE)` kullanın ve `StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)` ile beş katlı çapraz doğrulamanın ortalama AUC değerini bildirin. Sonucu üç ondalık basamağa yuvarlayın.

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma J", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
fixed = [f for f in features if f != "followup_months"]
forest_cv = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=RANDOM_STATE)
folds = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
answer = round(cross_val_score(forest_cv, registry[fixed], registry["event_12m"], cv=folds, scoring="roc_auc").mean(), 3)
teach.check("Alıştırma J", answer)

Etkileşimli labdaki [prompt atölyesinde](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/lab.html#part-f) böyle bir görevin promptu, prompt kartının yedi alanıyla yazılır. İki sorunu da yakalayacak denetim *Test* alanına yazılır: Modelin görmediği hastalarda sınanması.

## 10. Uygulama anahtarı: Başka bir sonuç, başka değişkenler

`OUTCOME` ve `VARIABLES` değerlerini değiştirin ve hücreyi yeniden çalıştırın. Hücre, bir lojistik regresyonun ve bir rastgele ormanın çapraz doğrulanmış AUC değerini ve lojistik regresyonun odds oranlarını bildirir. Denemeye değer sonuçlar şunlardır: `"cad"`, `"event_12m"`, `"af"` ve `"event"`. Her biri için önce karar anında hangi değişkenlerin bilinebileceğini düşünün.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 10. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB02_makine_ogrenmesi_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(10)

In [ ]:
OUTCOME = "af"
VARIABLES = ["age", "bmi", "hypertension", "lvef", "lavi", "ntprobnp"]

Xo = registry[VARIABLES].copy()
if "ntprobnp" in Xo:
    Xo["ntprobnp"] = np.log(Xo["ntprobnp"])
yo = registry[OUTCOME]
folds = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
lr_o = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
rf_o = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=RANDOM_STATE)
print("Prevalans:", OUTCOME, round(yo.mean(), 3))
for name, m in [("lojistik regresyon", lr_o), ("rastgele orman", rf_o)]:
    s = cross_val_score(m, Xo, yo, cv=folds, scoring="roc_auc")
    print(f"{name}: AUC {s.mean():.3f} ± {s.std():.3f}")
lr_o.fit(Xo, yo)
pd.Series(np.exp(lr_o.named_steps["logisticregression"].coef_[0]), index=VARIABLES,
          name="SS başına odds oranı").round(2)

## Alanlara göre uygulama görevleri

| Alan | Görev |
|---|---|
| Kalp yetersizliği | Kendi kliniğinizin protokolündeki izlem NT-proBNP değerini bir düşünce deneyi olarak hesaba katın: Fenogrupları yatak başında daha kullanışlı kılmak için dokuz kümeleme değişkeninden hangisini değiştirirdiniz? Kendi seçtiğiniz değişkenlerle yeniden kümeleyin ve Kaplan-Meier eğrilerini karşılaştırın. |
| Girişimsel kardiyoloji | 5. bölümdeki değişkenlerle kayıttan bir koroner arter hastalığı modeli kurun; hastalığı olmayan hastaların en fazla yüzde 20'sini anjiyografiye gönderecek eşiği seçin ve bu eşikteki duyarlılığı bildirin. |
| Elektrofizyoloji | Atriyal fibrilasyonu (`af`) yaş, BKİ, hipertansiyon, sol atriyum hacim indeksi ve NT-proBNP ile öngörün. Bilginin çoğunu hangi değişken taşımaktadır ve rastgele orman, lojistik regresyonun ötesinde bir katkı sağlıyor mu? |
| Kardiyovasküler görüntüleme | Kalp yetersizliği olan hastaları yalnızca ekokardiyografik değişkenlerle (`lvef`, `lavi`, `e_e_prime`, `lv_mass_index`, `tapse_mm`) kümeleyin ve gizli fenogruplarla uyumu dokuz değişkenli çözümle karşılaştırın. |
| Koruyucu kardiyoloji | Diğer değerleri ortalama düzeyde olan, sigara içen ve içmeyen iki kişi için `cad_model` modelinin verdiği koroner arter hastalığı olasılığını yaşa karşı çizin. Bu modelde sigaranın etkisi kaç yaşlık bir farka eşittir? |
| Kardiyo-onkoloji | Duyarlılığı 0,90 ve özgüllüğü 0,80 olan bir kardiyotoksisite tarama modeli düşünün. Yüzde 1 ile 20 arasındaki prevalanslar için PPV değerini çizin ve modelin hangi prevalanstan itibaren kullanılmaya değer olacağına karar verin. |
| Yoğun bakım | Sınıf ağırlıklı 12 aylık olay modelini kullanın ve olayların yüzde 80'ini bulan eşiği seçin. Model her 100 hastada kaç alarm verir ve bunların kaçı gerçek bir olaydır? |

## Defterden sonra

[Etkileşimli labı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/lab.html) açın ve *Kendini sına* sekmesindeki öz değerlendirmeyi yapın. Ardından *Düşün ve dışa aktar* sekmesinde yansıtma metnini yazın ve öğrenme günlüğünü dışa aktarın. [Hafta sayfasında](README.md) anlatılan 2. haftanın haftalık projesi, 7. bölümdeki kümelerden ve 8. bölümdeki iki fonksiyondan başlar.

<div align="center">

---

**Prof. Dr. Utku Köse**<br>
Profesör, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi, Isparta, Türkiye<br>
Kurucu Müdür, Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM), Süleyman Demirel Üniversitesi<br>
Bilgisayar Bilimleri Anabilim Dalı Başkanı, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi<br>
Diğer kurumlar: University of North Dakota (ABD), Universidad Panamericana (Meksiko, Meksika), Vel Tech University (Chennai, Hindistan)<br>
IEEE Kıdemli Üyesi, ACM Profesyonel Üyesi

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı, Türk Kardiyoloji Derneği. İlk sürüm: Kasım 2026. İçerik CC BY 4.0, kod MIT.*

</div>